In [20]:
from kafka import KafkaConsumer
from models import Ride, ride_deserializer

server = 'localhost:9092'
topic_name = 'green-trips'

consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-to-postgres',
    value_deserializer=ride_deserializer
)

In [21]:
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)
conn.autocommit = True
cur = conn.cursor()

In [ ]:
from datetime import datetime

print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
for message in consumer:
    ride = message.value 
    try:   
        cur.execute(
            """INSERT INTO green_trips
            (lpep_pickup_datetime, lpep_dropoff_datetime,  PULocationID, DOLocationID, 
            passenger_count, trip_distance , tip_amount, total_amount)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)""",
            (ride.lpep_pickup_datetime, ride.lpep_dropoff_datetime, 
            ride.PULocationID, ride.DOLocationID, 
            ride.passenger_count, ride.trip_distance, 
            ride.tip_amount, ride.total_amount)
        )
    except:
        print(f"Error inserting ride: {ride}")

    count += 1
    if count % 100 == 0:
        print(f"Inserted {count} rows...")

consumer.close()
cur.close()
conn.close()

Listening to green-trips and writing to PostgreSQL...
